In [ ]:
import requests
import pandas as pd
import io

def search_stations():
    # Targeted collection for station metadata
    url = "https://api.weather.gc.ca/collections/climate-stations/items"
    
    prov = input("Enter Province Code (e.g., BC, ON): ").upper().strip()
    name_query = input("Station name contains: ").upper().strip()

    # CQL filter for server-side searching
    cql_filter = f"PROV_STATE_TERR_CODE='{prov}' AND STATION_NAME LIKE '%{name_query}%'"
    
    params = {
        "lang": "en",
        "filter": cql_filter,
        "limit": 100
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        # Parse GeoJSON into a flat list for Pandas
        data = response.json()
        stations = [f['properties'] for f in data.get('features', [])]
        
        if not stations:
            print("No stations found.")
            return None

        df = pd.DataFrame(stations)
        print(f"\nFound {len(df)} stations:")
        print(df[['STATION_NAME', 'CLIMATE_IDENTIFIER', 'PROV_STATE_TERR_CODE']].head())
        return df

    except Exception as e:
        print(f"Search failed: {e}")